# Generative Protein LLMs: Designing Enzymes from a Chemical Reaction

## Learning Outcomes

By the end of this notebook you will be able to:

- Explain the encoder--decoder transformer architecture and how it differs from the encoder-only and decoder-only variants seen in nb02 and nb03.
- Describe autoregressive generation and how the sampling hyperparameters (`temperature`, `top_k`, `top_p`, `repetition_penalty`) shape the output.
- Write (reaction) SMILES strings for enzyme-catalyzed reactions.
- Use REXzyme, an encoder--decoder protein language model, to generate enzyme sequences for a target chemical reaction.
- Score generated designs with conditional perplexity, and explain which additional filters (structure, biophysics, function) turn millions of designs into a wet-lab shortlist.

---

**Credits.** This notebook is adapted from the FEBS Practical Course 2026 *"Generative AI for protein engineering"* teaching materials by Núria Mimbrero and Noelia Ferruz ([AI4PDLab/teaching](https://github.com/AI4PDLab/teaching)):

- Transformer theory (Section "The transformer, revisited") adapted from [`01_protein_language_models_protgpt2.ipynb`](https://github.com/AI4PDLab/teaching/blob/main/notebooks/01_protein_language_models_protgpt2.ipynb), sections 2.3-2.5.
- Enzyme / EC / Rhea-BRENDA / conditional-generation / SMILES background adapted from [`02_conditioning_enzyme_generation.ipynb`](https://github.com/AI4PDLab/teaching/blob/main/notebooks/02_conditioning_enzyme_generation.ipynb).
- The hands-on REXzyme example adapted from [`04b_rexzyme_pretrained.ipynb`](https://github.com/AI4PDLab/teaching/blob/main/notebooks/04b_rexzyme_pretrained.ipynb).
- The design-filtering theory adapted from [`05_filtering_protein_designs.ipynb`](https://github.com/AI4PDLab/teaching/blob/main/notebooks/05_filtering_protein_designs.ipynb).

## Install and import required packages

In [ ]:
!pip -q install transformers torch torchvision torchaudio

In [ ]:
import math

import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

## From reading proteins to writing them

In the first notebooks of this series we used protein LLMs as *readers*:

- **nb01** - we tokenized biological text and saw that a model only ever sees integer token IDs.
- **nb02** - we ran ESM-2, an **encoder-only** model, and watched self-attention mix context into every residue's embedding.
- **nb03 / nb03b** - we bolted a classification head onto an encoder trunk and fine-tuned it (fully, and with LoRA) to *label* promoter sequences.

Every one of those models **consumed** a sequence. A generative protein language model does the opposite: it **writes** sequences, one token at a time, in a language it learned from millions of natural proteins.

A useful analogy - proteins have a grammar. Like sentences, they are chains of symbols (the 20 amino acids) whose order determines meaning (fold and function). Natural proteins are "sentences" that evolution has proofread for billions of years; a model trained on them learns which next tokens are grammatical. Once we can compute probabilities over this grammar, we can *sample* brand-new "sentences" no organism has ever written - potentially with a function we ask for.

In this notebook we go one step beyond free text generation: we ask a model to write an enzyme for a **specific chemical reaction**, specified in the language of chemistry (SMILES).

## The transformer, revisited

### Architecture of the transformer

In nb02 we peeked inside ESM-2's encoder layers. The *original* transformer (Vaswani et al., 2017) is composed of **both** an encoder and a decoder - a design borrowed from machine translation, where the model reads a sentence in one language and writes it in another.

Before the transformer processes a sequence:

- we break the input into smaller chunks (**tokenization**),
- convert each token into a vector (**embedding**),
- and add **position encodings** so the model knows where each token sits in the sequence.

**The encoder** processes the embeddings through its multi-head self-attention and feed-forward sub-layers. Self-attention captures contextual information for each position; the feed-forward network refines it. The result is one vector per input token that captures its context with respect to the entire input - and these vectors are passed to the decoder.

**The decoder** takes the encoded sequence and generates the output **token by token**. Its extra ingredient is the **masked self-attention** sub-layer: when predicting a token, the decoder is prevented from looking at positions that have not been generated yet, so the prediction depends only on the tokens already produced. Each decoder layer also contains the two sub-layers known from the encoder (**multi-head attention and feed-forward**), with one difference: its attention is pointed at the *encoder's* output vectors, so the generation constantly consults the input.

![The original transformer architecture](../images/transformer_architecture.png)

Read this picture as a translation pipeline: **the encoder reads the source, the decoder writes the target**. Keep it in mind - you will meet it again in this notebook, where the "source language" is chemistry and the "target language" is proteins.

### Encoder-only, decoder-only - and both

The three transformer families you have met in this course are variations of the original design:

| Variant | Part used | Trained to | Good for | You have seen |
|---|---|---|---|---|
| **Encoder-only** | encoder | reconstruct masked tokens (*masking*) | representations, classification | BERT/DNABERT, **ESM-2 (nb02)**, the nb03 trunk |
| **Decoder-only** | decoder | predict the next token (*autoregressive*) | generation | GPT, ProtGPT2 |
| **Encoder-decoder** | both | translate input -> output | sequence-to-sequence tasks | T5, **REXzyme (today)** |

![Encoder-only vs decoder-only versions of the transformer](../images/encoder_decoder_variants.png)

Encoder-only models are excellent *readers*: they see the whole sequence at once, so their representations are context-aware - but they cannot naturally write long new sequences. Decoder-only models *write* fluently but must be steered through prompts or tags. Encoder-decoder models are the translation specialists: one modality in, another modality out - exactly today's task, where the input is a chemical reaction and the output is a protein.

### How generation actually works

A decoder generates autoregressively. Starting from a special beginning-of-sequence token (`<BOS>`) - or from an amino-acid prefix you provide - the model predicts a probability distribution over the next token, samples one, appends it, and repeats. A causal attention mask ensures every prediction depends only on the preceding tokens, so the probability of a whole sequence factorizes into next-token conditionals:

$$p(X)=\prod_{i=1}^{N} p(x_i \mid x_{<i})$$

Two ways to use a trained generator:

- **Unconditional generation**: start from the `<BOS>` token only. The model free-runs through protein sequence space - useful when you simply want "plausible proteins".
- **Prompted generation**: provide an amino-acid prefix and the model writes a plausible *continuation* (the trick used in the source course with ProtGPT2).

Sampling is what creates diversity - the model does not always pick the single most probable token. The knobs (introduced with ZymCTRL in the source course, and used hands-on below):

| Hyperparameter | Effect |
|---|---|
| `do_sample=True` | activate sampling (instead of always taking the argmax) |
| `temperature` | divides the logits before softmax: low -> conservative, high -> diverse but riskier |
| `top_k` | sample only from the `k` most probable candidates (0 = disabled) |
| `top_p` | sample from the smallest set of tokens with cumulative probability >= `p` (1.0 = disabled) |
| `repetition_penalty` | down-weights tokens already used |
| `max_length` / `max_new_tokens` | hard limit on output length |
| `num_return_sequences` | how many candidates to draw from the same prompt |

One warning, carried over from the source material, is the entire motivation for the filtering section below: **a high-probability sequence merely looks natural - it does not guarantee the protein folds or performs a function.** Probability is a similarity-to-nature score, not a wet-lab assay, so generative pipelines always end with filters (perplexity, structure prediction, biophysics).

In an encoder-decoder model like REXzyme the same autoregressive loop runs inside the decoder, conditioned on the encoder's reading of the input: the model writes *in protein* what the reaction asks *in chemistry*.

## The task: design an enzyme for a chemical reaction

We will generate enzymes for a target reaction. Three pieces of background first: what enzymes are (so the arrow below means something), how reactions are labeled and stored (so we know where training data comes from), and how molecules are written as strings (so we can craft our prompts by hand).

### Enzymes in one minute

Enzymes are nature's catalysts: (usually) proteins that accelerate chemical reactions under mild conditions - ambient temperature, neutral pH, water. That is exactly why industrial biotechnology loves them: greener processes with fewer harsh reagents (the source course lists acrylamide production, vanillin synthesis, the antiviral islatravir and the sweetener aspartame among its real-world examples).

The chemistry happens at the **active site**, a small pocket shaped and chemically patterned to bind specific **substrates**. Many active sites also recruit non-protein helpers: **cofactors** (inorganic ions such as Mg2+ or Zn2+) and **coenzymes** (organic carriers such as NAD+ or FAD).

Catalysis in three beats: **(1)** the substrates bind the active site, **(2)** bonds break and form, **(3)** the products are released and the enzyme is ready again.

The physical trick is always the same: the active site stabilizes the reaction's **transition state**, lowering the activation free-energy barrier in the rate law

$$k \propto e^{-\Delta G^{\ddagger}/RT}$$

so the reaction runs faster. At 25 C, reducing the barrier by roughly 5.7 kJ/mol buys a tenfold rate increase. Note what enzymes do *not* change: the reaction's overall free-energy difference - only how fast equilibrium is reached.

Every generated design you will score below ultimately answers the question: *does this sequence arrange the right residues into a pocket that catalyzes this reaction?*

### EC numbers, Rhea and BRENDA: where the training data comes from

We cannot ask a model for "an enzyme that does this" without a shared vocabulary.

- **EC numbers** label each enzyme class with four progressively more specific digits (`class.subclass.sub-subclass.serial`), e.g. EC 4.2.1.1 = carbonic anhydrase. The seven top-level classes: **1 oxidoreductases, 2 transferases, 3 hydrolases, 4 lyases, 5 isomerases, 6 ligases, 7 translocases**.
- **Rhea** (https://www.rhea-db.org) is an expert-curated database of *enzyme-catalyzed reactions*: every entry is a chemically balanced equation (defined at pH 7.3), linked to ChEBI compounds and to the UniProt proteins that catalyze it.
- **BRENDA** (https://www.brenda-enzymes.org) is the sister resource: curated *functional data* per enzyme (kinetics, substrates, conditions) across ~8,476 EC labels.

A reaction-enzyme **pair** - a balanced equation plus a catalyzing sequence - is exactly what a supervised model needs to learn translation from chemistry to protein.

### Conditional generation: steering the model

Unconditional generation explores sequence space, but we want to *condition* function. Two design patterns:

1. **Condition as a tag in the input stream (decoder-only).** Prepend a label and let the model autoregress on it. ZymCTRL is a CTRL-based decoder trained on 37M UniProt sequences with EC tags (prompts shaped like `4.2.1.1<sep><start>...<end>`): it generates sequences carrying the requested EC activity, and its experimentally validated designs included lactate dehydrogenases with 40-50% identity to natural ones.
2. **Condition through an encoder (encoder-decoder).** The reaction is *read* by the encoder, and the decoder translates it into protein. This is **REXzyme**.

**REXzyme** (REAction-ENZYMES) is an encoder-decoder transformer trained on ~21M reaction-sequence pairs harvested from Rhea, BRENDA, BKMS and UniProtKB. Its published designs were ordered, globular proteins - distant from natural sequences - that matched the catalytic activity of the reaction prompt: a generalization claim, not just sequence interpolation.

### SMILES: molecules as sentences

**SMILES** (Simplified Molecular Input Line Entry System) writes a molecular structure as a plain ASCII string - chemistry made tokenizer-friendly. For a friendly reference beyond this crash course, see this [SMILES notation guide](https://www.moldraw.com/tools/free-chem-tools/smiles-notation-guide.html). The rules:

- Atoms appear as their symbols (`C`, `N`, `O`, ...); **single bonds are implicit**, `=` double, `#` triple, `:` aromatic.
- **Branches** go in parentheses: `CC(=O)O` writes acetic acid (methyl + carbonyl + hydroxyl).
- **Rings** are opened with matching ring-closure numbers.
- **Aromatic** atoms are lowercase (`c1ccccc1` = benzene).
- Hydrogens are **implicit** (only written when meant explicitly).
- **Charges** live in brackets with `+`/`-`: `[O-]`, `[NH4+]`.
- **Stereochemistry**: `@`/`@@` for chirality, `/` and `\` for cis/trans double bonds.

As in natural language, one molecule has many valid SMILES strings (you can read the structure starting from any atom); algorithms pick one **canonical SMILES** per molecule - and the redundancy is even useful as data augmentation.

Decode these with the rules above as a warm-up: `O` is water, `O=C=O` is carbon dioxide, `O=C([O-])O` is bicarbonate. Keep them in mind - they are about to show up in a model prompt.

### Reaction SMILES: our prompt format

A **reaction SMILES** joins reactants and products with a double arrow, separate molecules in a set separated by a dot:

```
reactants >> products
```

That is the *entire prompt language* of REXzyme. Our running example is **carbonic anhydrase** (EC 4.2.1.1), the enzyme your blood uses to shuttle CO2:

```
O=C([O-])O.[H+]>>O.O=C=O
```

Read it aloud: bicarbonate + a proton -> water + carbon dioxide. The encoder tokenizes this chemistry; the decoder answers in amino acids.

![REXzyme, Figure 1 from the paper](../images/rexzyme_fig1.png)

*(REXzyme, Fig. 1: reaction SMILES in, enzyme out - trained on ~21M reaction-enzyme pairs from Rhea, BRENDA, BKMS and UniProtKB.)*

## Hands-on: generating enzymes for carbonic anhydrase

The code below is adapted from the course's [`04b_rexzyme_pretrained.ipynb`](https://github.com/AI4PDLab/teaching/blob/main/notebooks/04b_rexzyme_pretrained.ipynb).

> **Runtime note.** The REXzyme checkpoint is about 2.9 GB and downloads automatically from Hugging Face the first time you run this section. The original course runs it on a Colab T4 GPU (this notebook's metadata is set to GPU accordingly); CPU works but generation is slow.

### Load the two tokenizers

REXzyme speaks two languages, so it needs two vocabularies: one for chemistry (an SMILES byte-pair tokenizer) and one for proteins (amino acids). Both live in the [AI4PD/REXzyme](https://huggingface.co/AI4PD/REXzyme) repo, in separate subfolders.

In [ ]:
# Molecule / SMILES tokenizer (reads the reaction)
tokenizer_mol = AutoTokenizer.from_pretrained(
    "AI4PD/REXzyme",
    subfolder="tokenizer_aa--ABPE_SMILES/tokenizer_ABPE_rexzyme_offset",
)

# Amino-acid tokenizer (writes the enzyme)
tokenizer_aa = AutoTokenizer.from_pretrained(
    "AI4PD/REXzyme",
    subfolder="tokenizer_aa",
)

### Load the model

`AutoModelForSeq2SeqLM` is the sequence-to-sequence class - encoder and decoder in one object. We load the checkpoint used by the course (`checkpoint-90000`), move it to the GPU, and switch the model to `eval()` mode: dropout off, since we only want to sample.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForSeq2SeqLM.from_pretrained(
    "AI4PD/REXzyme",
    subfolder="checkpoint-90000",
)
model.to(device)
model.eval()
print(device)

### Generate candidate enzymes

Tokenize the reaction with the **chemical** tokenizer, generate with the course's sampling recipe (`top_k=100`, `top_p=1.0`, no repetition penalty, `do_sample=True`), draw `num_return_sequences=5` designs in a single call, and decode them with the **protein** tokenizer.

In [ ]:
reaction_smiles = "O=C([O-])O.[H+]>>O.O=C=O"  # carbonic anhydrase

inputs = tokenizer_mol(reaction_smiles, return_tensors="pt").to(device)

outputs = model.generate(
    **inputs,
    max_length=1024,         # longest possible design
    top_k=100,               # sampling parameters
    top_p=1.0,
    repetition_penalty=1.0,
    do_sample=True,
    num_return_sequences=5,  # number of designs
)

enzymes = [tokenizer_aa.decode(o, skip_special_tokens=True) for o in outputs]
for i, seq in enumerate(enzymes):
    print(f">design_{i} len={len(seq)}\n{seq}\n")

### Sanity pass

Before believing a single character, ask the cheap questions. Are the lengths in a plausible enzyme range (hundreds of residues - not five, not three thousand)? Are the five outputs *distinct* (identical strings would mean the sampling collapsed)? Is the alphabet amino acids only (no tokenization debris like digits or brackets)? The diversity comes from sampling: run the cell again with the same prompt and you get new designs.

What we still do **not** know: whether any of these fold, let alone whether they hydrolyze bicarbonate. Time to score; then, later, to filter.

### Scoring the designs: conditional perplexity

Perplexity is the exponentiated average negative log-likelihood a model assigns to a sequence - see the [Hugging Face perplexity docs](https://huggingface.co/docs/transformers/perplexity). For a conditional model we score the enzyme **given the reaction**, P(enzyme | reaction): we keep the reaction in the encoder, pass each candidate through the decoder in teacher-forcing mode, and ask for its loss - masking padding with the label `-100` so pads do not dilute the average - and finally take `exp(loss)`.

A low conditional perplexity means the model is confident that *this protein is an answer to this reaction*: the first, cheapest filter of the design pipeline. It is still a likelihood, not a verdict - use it to rank designs, never to accept them outright.

In [ ]:
def calculate_perplexity(model, src_ids, tgt_ids, tgt_tokenizer):
    """Conditional perplexity P(target | source), teacher-forcing mode."""
    labels = tgt_ids.clone()
    labels[labels == tgt_tokenizer.pad_token_id] = -100  # ignore padding
    with torch.no_grad():
        outputs = model(input_ids=src_ids, labels=labels)
        loss = outputs.loss
    return math.exp(loss.item())


# score the designs generated above (outputs are raw token tensors, one per design)
ppls = []
for i, seq in enumerate(outputs):
    tgt = seq.unsqueeze(0)  # add batch dimension
    ppl = calculate_perplexity(
        model, src_ids=inputs["input_ids"], tgt_ids=tgt, tgt_tokenizer=tokenizer_aa
    )
    ppls.append(ppl)
    print(f"Design {i} | len={len(enzymes[i])} | PPL = {ppl:.2f}")

best = min(range(len(ppls)), key=lambda i: ppls[i])
print(f"\nLowest perplexity: design {best}")

## From millions of designs to a wet-lab shortlist

This theory section is adapted from the course's [`05_filtering_protein_designs.ipynb`](https://github.com/AI4PDLab/teaching/blob/main/notebooks/05_filtering_protein_designs.ipynb) - no code, but arguably the most useful pipeline in the whole course.

Generative protein models can emit *millions* of candidates in an afternoon; validating one in the wet lab (clone it, express it, purify it, assay it) takes weeks and reagents. The goal of filtering is not to find the one perfect sequence - it is to **maximize the probability that whatever you do test is folded and functional**, so that scarce experiments go to the most promising designs. Which filters you use depends on the activity you are after; the common toolbox:

### Sequence-based filters (fast - run first)

- **Perplexity** (the one you computed above): exponentiated average NLL under a reference model. Low = the sequence statistics resemble natural proteins the model trusts; high = out-of-distribution. Keep the low tail (e.g., below the batch's 75th percentile).
- **ESM-1v variant-effect score**: for designs related to a known reference protein, score the mutations against the wild type with ESM-1v (fine-tuned for variant effects). Strongly negative scores flag positions the model considers *deleterious* - useful when your design looks like a decorated natural enzyme.
- **EC-label prediction (CLEAN)**: does a function-prediction model still assign your design the EC class you requested? A cheap sanity check that the conditioning "took".
- **Sequence similarity (MMseqs2 / BLAST)**: against natural databases - discard anything at 100% identity to a known protein (you did not design it) and usually track the novelty distance.

### Structure-based filters

- **pLDDT** is the per-residue confidence score (0-100) that AlphaFold or ESMFold attaches to each position: **>90 very high, 70-90 good, <70 unreliable**. For a design, look at the *average* (is the model confident the sequence has one stable structure?) and at long low-confidence stretches. ESMFold is free online for sequences up to 400 residues: https://esmatlas.com/about .
- **TM-score** measures fold similarity against a reference structure (computed with **Foldseek**): **>0.7 = same fold, 0.5-0.7 = partial similarity, <0.5 = different fold**. Use it when the design must *keep* (or deliberately *avoid*) a target architecture.

Sequence perplexity says "sounds natural"; structure scores say "folds into a defined shape - and that shape is the one I meant".

### Sequence-structure consistency (ProteinMPNN)

Structure predictors go sequence -> structure. **ProteinMPNN** runs the arrow backwards: given a backbone, it models what sequence builds it. Its score answers *"does this sequence fit this structure?"* - **a lower |score| means the sequence and the structure agree**. Pairing a forward check (pLDDT, TM-score) with this inverse check keeps only designs consistent in both directions.

### Biophysical sanity checks (cheap structure-level proxies)

- **SASA / hydrophobic exposure** - computed on the predicted structure (e.g., with Rosetta): a design that buries its apolar core exposes little hydrophobic surface to solvent; the course pipeline keeps `relative_hydrophobic_sasa < 3`. Exposed hydrophobic surfaces stick together - that is aggregation.
- **SAP** (spatial aggregation propensity) - same spirit; the pipeline keeps `average_sap` (the score divided by sequence length) <= 0.4.
- **Net charge** - count **+1** for K, R, H and **-1** for D, E. Designs with extreme net charge (say +40 or -50) are red flags: they prefer buffer conditions no cell provides.

### The function check: catalytic residues

All of the filters above certify *plausible folded protein*. Only one step certifies *could catalyze **your** reaction*: when the catalytic machinery is known (from a wild-type reference), verify that the key residues are present - directly in the sequence (search for the motif) or structurally (superimpose your prediction onto the wild type, e.g., in PyMOL, and measure the active-site geometry). A beautifully folded protein missing the one zinc-binding triad your reaction needs is a beautiful non-enzyme.

**Order matters** for cost: cheap sequence filters first (perplexity, EC prediction, similarity), then structure prediction (pLDDT, TM-score), then inverse folding, then biophysics, then the per-activity catalytic-residue check on the survivors. The course's example pipeline ran exactly: perplexity <= 75th percentile -> pLDDT > 70 -> TM-score > 0.8 -> not 100% identical to nature -> `average_sap` <= 0.4 -> `relative_hydrophobic_sasa` < 3 -> non-zero net charge -> EC-class match - and finally ranked the survivors by summing their |ESM-1v| and |ProteinMPNN| ranks.

## Exercise: write a prompt for an acrylamide factory

One of the green-chemistry case studies from the source course: **acrylamide**, an industrial monomer produced for decades at high temperature and pressure with copper catalysts, can instead be made by **nitrile hydratase** (EC 4.2.1.84) - the enzyme simply **adds a water molecule to acrylonitrile**, at ambient conditions, in water, without heavy metals (acrylamide polymers are used in water treatment, paper, mining and oil recovery).

The chemistry in words:

```
acrylonitrile + water  ->  acrylamide
```

Your task:

1. Express the reaction as a reaction SMILES (reactants and products separated by `>>`, separate molecules separated by `.`).
2. Generate 5 candidate nitrile hydratases with `tokenizer_mol` / `model.generate` using the **same** sampling parameters as the hands-on.
3. Score each design's conditional perplexity and rank them.
4. Propose which of the Section-5 filters you would run next - and say what might go wrong for this *specific* prompt (hint: how was the training data curated?).

### Hint

Build each molecule with the SMILES rules from the theory section. Acrylonitrile: a three-carbon chain, a vinyl group (`C=C`) then a nitrile (`C#N`). Acrylamide: the same vinyl group, then a carbonyl (`C=O`) whose carbon carries an amine (`N`) - written as a branch, so the terminal amine sits in parentheses. Water is just `O`.

#### Answer

One correct reaction SMILES:

```
C=CC#N.O>>C=CC(N)=O
```

- `C=CC#N` - acrylonitrile: vinyl `C=C` then nitrile `C#N`.
- `O` - water, the second reactant.
- `C=CC(N)=O` - acrylamide: vinyl, then the carbonyl carbon written `C(...)=O` with the amine `N` as its branch.

```python
reaction_smiles = "C=CC#N.O>>C=CC(N)=O"  # nitrile hydratase, EC 4.2.1.84
inputs_nh = tokenizer_mol(reaction_smiles, return_tensors="pt").to(device)

outputs_nh = model.generate(
    **inputs_nh,
    max_length=1024,
    top_k=100,
    top_p=1.0,
    repetition_penalty=1.0,
    do_sample=True,
    num_return_sequences=5,
)
enzymes_nh = [tokenizer_aa.decode(o, skip_special_tokens=True) for o in outputs_nh]

ppls_nh = [
    calculate_perplexity(model, inputs_nh["input_ids"], o.unsqueeze(0), tokenizer_aa)
    for o in outputs_nh
]
for i, (seq, ppl) in enumerate(zip(enzymes_nh, ppls_nh)):
    print(f"Design {i} | len={len(seq)} | PPL = {ppl:.2f}")
```

**What could go wrong (the instructive part).** Acrylonitrile is an *abiotic*, petrochemical substrate: REXzyme's training data comes mostly from curated biology (Rhea, BRENDA, BKMS, UniProtKB), so this prompt sits closer to the edge of the model's training distribution than the carbonic anhydrase example. Expect higher perplexities overall - and remember perplexities are comparable *within* one prompt family, not across reactions of different difficulty. Lean harder on the Section-5 filters (the EC-prediction check is especially relevant here), and treat the whole thing as a genuine out-of-distribution design experiment: results are hypotheses to test, not answers to trust.

## Annex: where this can go

- **Sampling sweeps.** Re-run the hands-on with `temperature` 0.7 / 1.0 and `top_k` 10 / 100 / 500; chart perplexity vs output length vs diversity. Watch a `repetition_penalty > 1` prune poly-residue tails.
- **Compare conditioning regimes.** ZymCTRL ([paper](https://doi.org/10.1101/2024.05.03.592223)) conditions on an EC *tag* (`3.5.5.1<sep><start>...<end>`) instead of a full reaction - try EC 3.5.5.1 (nitrilase, right next to the enzyme family of the exercise above) and compare the two conditioning styles on the same target.
- **Automate the structure filter.** The ESMFold API (`https://api.esmatlas.com/foldSequence/v1/pdb/`) takes a sequence and returns a PDB you can render in the notebook with `py3Dmol`; mean pLDDT can be extracted from the B-factors to close the generation -> structure-confidence loop without leaving Jupyter.
- **Run the full funnel.** Generate 100 sequences for one reaction, apply the Section-5 pipeline in order, and count the survivors at each step - the funnel shape *is* the experimental result.

### Resources

- Source course repo: https://github.com/AI4PDLab/teaching - REXzyme model: https://huggingface.co/AI4PD/REXzyme
- SMILES guide: https://www.moldraw.com/tools/free-chem-tools/smiles-notation-guide.html
- Perplexity: https://huggingface.co/docs/transformers/perplexity - Rhea: https://www.rhea-db.org - BRENDA: https://www.brenda-enzymes.org - ESMFold online (<=400 aa): https://esmatlas.com/about